In [1]:
import sys
import polars as pl
sys.path.insert(0, '..')
import time
import json
from fs_thesis import sql, show
from sklearn.metrics import classification_report, confusion_matrix
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
import warnings
from tqdm.auto import tqdm
from sklearn.metrics import (recall_score, precision_score, f1_score, 
                             accuracy_score, roc_auc_score, confusion_matrix)


In [2]:
# ── Run-Ordner (einmalig für das ganze Notebook) ──
_run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = Path(f"/Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_{_run_timestamp}")
PLOTS_DIR = RUN_DIR / "plots"
RESULTS_DIR = RUN_DIR / "results"
for d in [PLOTS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

_plot_counter = 0

def show_and_save(fig, name: str = None):
    """fig.show() + PNG speichern in den aktuellen Run-Ordner."""
    global _plot_counter
    _plot_counter += 1
    filename = name or f"plot_{_plot_counter:02d}"
    path = PLOTS_DIR / f"{filename}.png"
    fig.write_image(str(path), scale=2)
    print(f"💾 {path}")
    fig.show()

print(f"📁 Run-Ordner: {RUN_DIR}")

📁 Run-Ordner: /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260303_212527


# 1. Data Pipeline Overview
This notebook uses the centralized data pipeline defined in `fs_thesis.data_loader`. Below is a documentation of the logic encapsulated in `load_final_data()`.

### A. Patient Demographics (Baseline)
We extract the following baseline features from the first admission (`t0_time`):
- **Identifier**: `subject_id`
- **Demographics**: `gender`, `anchor_age`, `race`, `marital_status`, `language`, `insurance`
- **Context**: `admission_type`
- **BMI**: Median BMI from `hosp.omr` (Left Join, missing values are handled by TabPFN)

### B. Event Definition (Target)
The event is defined as the **first occurrence** of a specific ICD diagnosis (e.g., Heart Failure `I50%`).
- **Event Time**: Timestamp of the diagnosis.
- **Censoring**: If no event occurs, the patient is censored at the date of death (`dod`) or end of follow-up.

### C. Target Calculation Logic
The target variable is derived based on the time-to-event (`duration`):
1. **Calculate Duration**:
   - `t_event` = Days from baseline to diagnosis.
   - `t_death` = Days from baseline to death.
   - Priority: Event Time > Death Time > Fallback (2000 days).
   - Negative durations are clipped to 0.

2. **Define Classes (`target`)**:
   - **Class 0 (Early Event)**: Event occurs $\le$ 365 days.
   - **Class 1 (Late Event)**: Event occurs $>$ 365 days.
   - **Class 2 (Censored/Control)**: No event observed (censored or healthy).

In [3]:
from fs_thesis.data_loader import load_final_data
df_final = load_final_data()

In [4]:
# Prüfen wie viele Missings wir haben
print("Missing BMI from Join, before feature engineering: TabPFN will handle these missing values, but it's good to know how many we have.")
print(f"Missing BMI: {df_final['bmi'].null_count()} of {len(df_final)}")

Missing BMI from Join, before feature engineering: TabPFN will handle these missing values, but it's good to know how many we have.
Missing BMI: 98306 of 223452


# 2. Preprocessing
## Splitting

In [5]:
from fs_thesis.preprocessing import preprocess_data, balance_data, get_X_y
df_train, df_val, df_test = preprocess_data(df_final)

Shapes -> Train: (143008, 17), Val: (35753, 17), Test: (44691, 17)


## Sampling (Balancing)

In [6]:
# Balance (only for train data!)
df_balanced = balance_data(df_train, n_samples=3000)

# split Features & Target (all Sets!)
X_train, y_train = get_X_y(df_balanced)
X_val, y_val = get_X_y(df_val)
X_test, y_test = get_X_y(df_test)

# Jetzt passt auch der fucking Print
print(f"Train (balanced): {len(y_train)} | Val (real): {len(y_val)} | Test (real): {len(y_test)}")

Train (balanced): 9000 | Val (real): 35753 | Test (real): 44691


# 3. Training
## Classifier

In [7]:
from tabpfn import TabPFNClassifier
classifier = TabPFNClassifier(device='mps') # Zurück auf CPU, für schnellere Vorhersagen bei kleinen N

## Fit

In [8]:
print("Start fitting...")
classifier.fit(X_train, y_train)
print("Training done!")

Start fitting...
Training done!


## Predict

In [9]:
# 3. Vorhersage (Validierung)
X_val_sample = X_val.iloc[:1000]
y_val_sample = y_val[:1000]

# Sample run (faster)
sample = True
# For importance analysis
skip = False

# --- Aktive Variablen ---
X_val_active = X_val_sample if sample else X_val
y_val_active = y_val_sample if sample else y_val

print("Start predict (3min)...")
y_val_pred = classifier.predict(X_val_active)
print("Done!")

Start predict (3min)...
Done!


# 4. Validation (val_set for optimizing)

In [10]:
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

print("Evaluating (3min)...")
y_proba = classifier.predict_proba(X_val_active)
roc_auc = roc_auc_score(y_val_active, y_proba, multi_class='ovr', average='macro')

Evaluating (3min)...


In [11]:
print(f"Accuracy: {accuracy_score(y_val_active, y_val_pred):.2f}")
print(f"AUC - ROC Score: {roc_auc:.2f}")
print("\nClassification Report:")
print(classification_report(y_val_active, y_val_pred, 
                            target_names=['early (<1J)', 'late (>1J)', 'healthy']))

Accuracy: 0.56
AUC - ROC Score: 0.83

Classification Report:
              precision    recall  f1-score   support

 early (<1J)       0.14      0.70      0.23        40
  late (>1J)       0.10      0.78      0.17        36
     healthy       0.99      0.55      0.70       924

    accuracy                           0.56      1000
   macro avg       0.41      0.67      0.37      1000
weighted avg       0.92      0.56      0.66      1000



# 4.1 Visualization Data

In [12]:
import plotly.express as px

df_analyze = X_val_active.copy()
if hasattr(df_analyze, "to_pandas"):
    df_analyze = df_analyze.to_pandas()

fig = px.box(df_analyze, x="insurance", y="anchor_age", 
             title="'Medicare'-Effekt, be causion with age for TabPFN",
             points="all", 
             color="insurance")

show_and_save(fig, "medicare_age_effect")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260303_212527/plots/medicare_age_effect.png


In [13]:
risk_score = y_proba[:, 0]

df_analyze['risk_score'] = risk_score
df_analyze['true_label'] = y_val_active

### BMI Analyse

In [14]:
bins = [0, 18.5, 25, 30, 100]
labels = ['Untergewicht (<18.5)', 'Normal (18.5-25)', 'Übergewicht (25-30)', 'Adipositas (>30)']
df_analyze['bmi_group'] = pd.cut(df_analyze['bmi'], bins=bins, labels=labels)

df_bmi_mean = df_analyze.groupby('bmi_group', observed=True)['risk_score'].mean().reset_index()

fig1 = px.bar(
    df_bmi_mean, 
    x='bmi_group', 
    y='risk_score',
    text_auto='.1%', # Zeigt %-Wert direkt auf dem Balken
    title='Avg risk by BMI-Group',
    labels={'risk_score': 'Risk Score', 'bmi_group': 'BMI Group'},
    color='risk_score',
    color_continuous_scale='Reds',
    template="plotly_white"
)
fig1.update_layout(yaxis_tickformat='.0%') # Y-Achse als Prozent formatieren
show_and_save(fig1, "risk_by_bmi")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260303_212527/plots/risk_by_bmi.png


### Age Analyse

In [15]:
df_analyze['age_group'] = pd.cut(
    df_analyze['anchor_age'], 
    bins=[0, 20, 30, 40, 50, 60, 70, 80, 90, 120],  # 0-20, 20-30, ..., 90-120
    labels=['<20', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80-89', '90+']
)
df_analyze['age_group'] = df_analyze['age_group'].astype(str)

df_age_mean = df_analyze.groupby('age_group')['risk_score'].mean().reset_index()

fig2 = px.bar(
    df_age_mean, 
    x='age_group', 
    y='risk_score',
    text_auto='.1%',
    title='Risk by Age Group',
    labels={'risk_score': 'Risk Score', 'age_group': 'Age Group'},
    color='risk_score',
    color_continuous_scale='Reds',
    template="plotly_white"
)
fig2.update_layout(yaxis_tickformat='.0%')
show_and_save(fig2, "risk_by_age")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260303_212527/plots/risk_by_age.png


### Social - Economy

In [16]:
df_ins_mean = df_analyze.groupby(['insurance', 'gender'])['risk_score'].mean().reset_index()

fig3 = px.bar(
    df_ins_mean, 
    x='insurance', 
    y='risk_score', 
    color='gender', 
    barmode='group',
    text_auto='.1%',
    title='Risk by Gender and Insurance',
    labels={'risk_score': 'Risk Score', 'insurance': 'Insurance'},
    template="plotly_white"
)
fig3.update_layout(yaxis_tickformat='.0%')
show_and_save(fig3, "risk_by_gender_insurance")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260303_212527/plots/risk_by_gender_insurance.png


### Language Analyse

In [25]:
df_lang_mean = df_analyze.groupby('language')['risk_score'].mean().reset_index()
df_lang_mean = df_lang_mean.sort_values('risk_score', ascending=True)

fig4 = px.bar(
    df_lang_mean, 
    x='language', 
    y='risk_score',
    text_auto='.1%',
    title='Avg Risk by Language',
    labels={'risk_score': 'Risk Score', 'language': 'Language'},
    color='risk_score',
    color_continuous_scale='Reds',
    template="plotly_white"
)
fig4.update_layout(yaxis_tickformat='.0%')
show_and_save(fig4, "risk_by_language")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260303_212527/plots/risk_by_language.png


### Marital Status Analyse

In [26]:
df_marital_mean = df_analyze.groupby('marital_status')['risk_score'].mean().reset_index()
df_marital_mean = df_marital_mean.sort_values('risk_score', ascending=True)

fig5 = px.bar(
    df_marital_mean, 
    x='marital_status', 
    y='risk_score',
    text_auto='.1%',
    title='Avg Risk by Marital Status',
    labels={'risk_score': 'Risk Score', 'marital_status': 'Marital Status'},
    color='risk_score',
    color_continuous_scale='Reds',
    template="plotly_white"
)
fig5.update_layout(yaxis_tickformat='.0%')
show_and_save(fig5, "risk_by_marital_status")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260303_212527/plots/risk_by_marital_status.png


### Race Analyse

In [27]:
df_race_mean = df_analyze.groupby('race')['risk_score'].mean().reset_index()
df_race_mean = df_race_mean.sort_values('risk_score', ascending=True)

fig6 = px.bar(
    df_race_mean, 
    x='race', 
    y='risk_score',
    text_auto='.1%',
    title='Avg Risk by Race',
    labels={'risk_score': 'Risk Score', 'race': 'Race'},
    color='risk_score',
    color_continuous_scale='Reds',
    template="plotly_white"
)
fig6.update_layout(yaxis_tickformat='.0%', xaxis_tickangle=45)
show_and_save(fig6, "risk_by_race")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260303_212527/plots/risk_by_race.png


### Admission Type Analyse

In [28]:
df_adm_mean = df_analyze.groupby('admission_type')['risk_score'].mean().reset_index()
df_adm_mean = df_adm_mean.sort_values('risk_score', ascending=True)

fig7 = px.bar(
    df_adm_mean, 
    x='admission_type', 
    y='risk_score',
    text_auto='.1%',
    title='Avg Risk by Admission Type',
    labels={'risk_score': 'Risk Score', 'admission_type': 'Admission Type'},
    color='risk_score',
    color_continuous_scale='Reds',
    template="plotly_white"
)
fig7.update_layout(yaxis_tickformat='.0%', xaxis_tickangle=45)
show_and_save(fig7, "risk_by_admission_type")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260303_212527/plots/risk_by_admission_type.png


# 4.2 Visualisation Prediction Performance

In [17]:
cm = confusion_matrix(y_val_active, y_val_pred)
cm_perc = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

labels = ['early (<1J)', 'late (1-3J)', 'healthy']

annot_text = [
    [f"<b>{val}</b><br>({perc:.1%})" for val, perc in zip(row_val, row_perc)]
    for row_val, row_perc in zip(cm, cm_perc)
]

fig = ff.create_annotated_heatmap(
    cm_perc, 
    x=labels, 
    y=labels, 
    annotation_text=annot_text, 
    colorscale='Reds'
)
fig.update_layout(
    title=f'Validation Check: Confusion Matrix ({len(y_val_active)} Samples)',
    xaxis_title="Vorhersage des Modells",
    yaxis_title="Tatsächlicher Verlauf (MIMIC-Daten)",
    template="plotly_white",
    height=600
)

show_and_save(fig, "confusion_matrix")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260303_212527/plots/confusion_matrix.png


In [18]:
import plotly.graph_objects as go

cm = confusion_matrix(y_val_active, y_val_pred)

label_list = [
    "real: early", "real: late", "real: healthy",
    "predicted: early", "predicted: late", "predicted: healthy"
]

source = [0, 0, 0, 1, 1, 1, 2, 2, 2]
target = [3, 4, 5, 3, 4, 5, 3, 4, 5]
value = cm.flatten()

color_link = [
    'rgba(255, 90, 90, 0.4)', 'rgba(255, 90, 90, 0.2)', 'rgba(255, 90, 90, 0.1)',
    'rgba(255, 127, 14, 0.2)', 'rgba(255, 127, 14, 0.4)', 'rgba(255, 127, 14, 0.1)',
    'rgba(44, 160, 44, 0.1)', 'rgba(44, 160, 44, 0.1)', 'rgba(44, 160, 44, 0.4)'
]

fig = go.Figure(data=[go.Sankey(
    node = dict(
      pad = 15, thickness = 20, line = dict(color = "black", width = 0.5),
      label = label_list, color = "grey"
    ),
    link = dict(
      source = source, target = target, value = value, color = color_link
  ))])

fig.update_layout(title_text="Patient-Flow: Reality vs. Predicted", font_size=12)
show_and_save(fig, "sankey_patient_flow")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260303_212527/plots/sankey_patient_flow.png


# 6 Robustness Check

In [ ]:
# --- Configuration ---
N_LOOPS = 20
RUN_NAME = "robustness_overnight"

import warnings, time, json
from tqdm.auto import tqdm
from sklearn.metrics import recall_score, precision_score

warnings.filterwarnings('ignore', message='.*Running on CPU with more than 200 samples.*')

log_file = RUN_DIR / "run.log"

def log(msg):
    print(msg)
    with open(log_file, 'a') as f:
        f.write(f"{datetime.now().strftime('%H:%M:%S')} | {msg}\n")

configs = [
    {"name": "ensemble_8",  "N_ensemble": 8,  "n_samples": 300},
    # {"name": "ensemble_32", "N_ensemble": 32,  "n_samples": 300}, --- IGNORE --- no difference to 8, but much slower
]

# Config speichern
json.dump({"n_loops": N_LOOPS, "configs": configs, "run_dir": str(RUN_DIR)},
          open(RUN_DIR / "config.json", "w"), indent=2)

log(f"RUN: {RUN_DIR.name} | Loops: {N_LOOPS} | Configs: {len(configs)}")

results = []
start_total = time.time()

for cfg_idx, config in enumerate(configs, 1):
    t0 = time.time()
    log(f"\n[{cfg_idx}/{len(configs)}] {config['name']} | ensemble={config['N_ensemble']}, samples={config['n_samples']}")
    config_results = []
    
    for i in tqdm(range(N_LOOPS), desc=f"[{cfg_idx}/{len(configs)}] {config['name']}"):
        try:
            try:
                clf = TabPFNClassifier(device='mps', n_estimators=config['N_ensemble'])
            except TypeError:
                clf = TabPFNClassifier(device='mps')
            
            df_bal = balance_data(df_train, n_samples=config['n_samples'], seed=42 + i)
            X_tr, y_tr = get_X_y(df_bal)
            clf.fit(X_tr, y_tr)
            y_pred = clf.predict(X_val)
            y_proba = clf.predict_proba(X_val)
            f1_pc = f1_score(y_val, y_pred, average=None)
            
            result = {
                'run_id': i, 'config_name': config['name'],
                'n_ensemble': config['N_ensemble'], 'n_samples': config['n_samples'],
                'accuracy': accuracy_score(y_val, y_pred),
                'roc_auc_macro': roc_auc_score(y_val, y_proba, multi_class='ovr', average='macro'),
                'f1_macro': f1_score(y_val, y_pred, average='macro'),
                'recall_macro': recall_score(y_val, y_pred, average='macro'),
                'precision_macro': precision_score(y_val, y_pred, average='macro'),
                'f1_class_0_early': f1_pc[0], 'f1_class_1_late': f1_pc[1],
                'f1_class_2_healthy': f1_pc[2], 'seed': 42 + i,
            }
            results.append(result)
            config_results.append(result)
        except Exception as e:
            log(f"  ⚠️ FEHLER Run {i}: {e}")

    elapsed = time.time() - t0
    if config_results:
        df_cfg = pd.DataFrame(config_results)
        log(f"  ✅ F1={df_cfg['f1_macro'].mean():.4f}±{df_cfg['f1_macro'].std():.4f} | {elapsed/60:.1f}min")
        pd.DataFrame(results).to_csv(RESULTS_DIR / f"checkpoint_config{cfg_idx}.csv", index=False)


RUN: tab_pfn_20260303_212527 | Loops: 20 | Configs: 3

[1/3] ensemble_8 | ensemble=8, samples=300


[1/3] ensemble_8:   0%|          | 0/20 [00:00<?, ?it/s]

  ✅ F1=0.3625±0.0099 | 91.5min

[2/3] ensemble_32 | ensemble=32, samples=300


[2/3] ensemble_32:   0%|          | 0/20 [00:00<?, ?it/s]

  ✅ F1=0.3624±0.0100 | 375.7min

[3/3] ensemble_64 | ensemble=64, samples=300


[3/3] ensemble_64:   0%|          | 0/20 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:

# Finale Ergebnisse
df_results = pd.DataFrame(results)
df_results.to_csv(RESULTS_DIR / "final_results.csv", index=False)
log(f"\nDONE in {(time.time()-start_total)/3600:.2f}h | {len(df_results)} runs")

# Summary Plots
df_summary = df_results.groupby('config_name').agg(
    f1_mean=('f1_macro','mean'), f1_std=('f1_macro','std'),
).reset_index().sort_values('f1_mean', ascending=True)

fig = px.bar(df_summary, x='f1_mean', y='config_name', error_x='f1_std',
    orientation='h', title='F1 Macro: Config Comparison', template='plotly_white', text_auto='.3f')
fig.update_layout(xaxis_tickformat='.0%', xaxis_title='F1 Macro', yaxis_title=None)
show_and_save(fig, "robustness_config_comparison")

fig2 = px.violin(df_results, x='config_name', y='f1_macro', box=True, points='all',
    title='F1 Macro Distribution per Config', template='plotly_white')
fig2.update_layout(yaxis_tickformat='.0%', xaxis_tickangle=45)
show_and_save(fig2, "robustness_f1_violin")

# Summary Table
df_summary_full = df_results.groupby('config_name').agg(
    f1_mean=('f1_macro','mean'), f1_std=('f1_macro','std'),
    acc_mean=('accuracy','mean'), acc_std=('accuracy','std'),
    auc_mean=('roc_auc_macro','mean'), auc_std=('roc_auc_macro','std'),
    f1_early_mean=('f1_class_0_early','mean'), f1_early_std=('f1_class_0_early','std'),
    f1_late_mean=('f1_class_1_late','mean'), f1_late_std=('f1_class_1_late','std'),
    f1_healthy_mean=('f1_class_2_healthy','mean'), f1_healthy_std=('f1_class_2_healthy','std'),
    n_runs=('run_id','count'),
).reset_index()
df_summary_full.to_csv(RESULTS_DIR / "summary_table.csv", index=False)

log(f"\n📁 {RUN_DIR} | plots: {len(list(PLOTS_DIR.glob('*.png')))} | results: {len(list(RESULTS_DIR.glob('*.csv')))}")


DONE in 9.01h | 41 runs
💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260303_212527/plots/robustness_config_comparison.png


💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260303_212527/plots/robustness_f1_violin.png



📁 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260303_212527 | plots: 8 | results: 4


# 6.1 Robustness Vizualiszation

In [21]:
# 4. Visualisierung der Robustness (Master-Thesis Style)
# Ziel: Durchschnittliche Performance zeigen + Stabilität (Fehlerbalken) beweisen

# Melt transformiert die Daten für Plotly (Wide -> Long Format)
import plotly.express as px
import os

df_melt = df_results.melt(
    id_vars=['run_id'], 
    value_vars=['f1_class_0_early', 'f1_class_1_late', 'f1_class_2_healthy'],
    var_name='Target Class', 
    value_name='F1 Score'
)

# Aggegierte Statistiken für Bar-Chart
df_stats = df_melt.groupby('Target Class')['F1 Score'].agg(['mean', 'std']).reset_index()

# Sortierung manuell festlegen (Logische Zeitreihe)
category_order = ['Früher Ausbruch (<1J)', 'Später Ausbruch (1-3J)', 'Kein Event / Gesund']
# Wir müssen die Werte in df_stats umbenennen, damit sie matchen, oder category_order anpassen
# Aktuell heißen die Werte 'f1_class_0_early' usw.
# Mapping Dictionary
name_mapping = {
    'f1_class_0_early': 'Früher Ausbruch (<1J)',
    'f1_class_1_late': 'Später Ausbruch (1-3J)',
    'f1_class_2_healthy': 'Kein Event / Gesund'
}
df_stats['Target Class'] = df_stats['Target Class'].map(name_mapping)
df_melt['Target Class'] = df_melt['Target Class'].map(name_mapping)

# 1. Bar Chart mit Error Bars (Klassisch wissenschaftlich)
fig = px.bar(
    df_stats, 
    x="Target Class", 
    y="mean", 
    error_y="std", # Zeigt die Standardabweichung als Antenne (Robustheit)
    title=f"Modell-Performance: Durchschnitt & Stabilität ({N_LOOPS} Runs)",
    text_auto='.1%', # Beschriftung direkt am Balken
    labels={'mean': 'Durchschnittlicher F1-Score'},
    color="Target Class", 
    # Pastell Farben wirken professioneller und weniger überladen
    color_discrete_sequence=px.colors.qualitative.Pastel,
    template="plotly_white",
    category_orders={"Target Class": category_order} # Erzwingt die logische zeitliche Reihenfolge
)

# Balken leicht transparent machen (0.8), damit sie nicht zu massiv wirken
fig.update_traces(marker_opacity=0.8, showlegend=False)

# 2. Scatter-Punkte darüber legen (Kontrastfarbe für bessere Sichtbarkeit)
# Zeigt jeden einzelnen Run als Punkt -> Maximale Transparenz der Ergebnisse
scatter_trace = px.strip(
    df_melt, 
    x="Target Class", 
    y="F1 Score", 
    category_orders={"Target Class": category_order}
).data[0]

# Styling der Punkte: Dunkles Kontrast-Blau mit weißem Rand
# Das sorgt für Lesbarkeit sowohl auf hellen als auch dunklen Hintergründen
scatter_trace.marker.color = '#34495e' # "Wet Asphalt" (Dunkles Grau-Blau)
scatter_trace.marker.size = 6
scatter_trace.marker.opacity = 0.8
scatter_trace.marker.line = dict(width=1, color='white') # Weißer Rand lässt Punkte "poppen"
scatter_trace.showlegend = False

fig.add_trace(scatter_trace)

# Achsen formatieren (Prozent statt 0.x)
fig.update_layout(
    yaxis_tickformat='.0%', 
    yaxis_title="F1 Score (Macro)", 
    xaxis_title=None, # X-Achsen Titel ist redundant wegen den Labels
    font=dict(size=14) # Schrift etwas größer für Thesis
)
fig.update_yaxes(range=[0, 1.1]) # Platz für Error Bars lassen

show_and_save(fig, "robustness_f1_per_class")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260303_212527/plots/robustness_f1_per_class.png


# 6.2 Test

In [22]:
# Nach dem Robustness Loop: Bestes Modell auf TEST-Set evaluieren
# Nimm den Median-Seed aus den Loop-Ergebnissen
best_seed = df_results.loc[df_results['f1_macro'].idxmax(), 'seed']
best_config = df_results.loc[df_results['f1_macro'].idxmax(), 'config_name']
log(f"Best config: {best_config}, seed: {int(best_seed)}")

# Reproduziere das beste Modell
df_bal_best = balance_data(df_train, n_samples=300, seed=int(best_seed))
X_tr_best, y_tr_best = get_X_y(df_bal_best)
clf_best = TabPFNClassifier(device='mps')
clf_best.fit(X_tr_best, y_tr_best)

# Finale Evaluation auf TEST (nicht Val!)
y_test_pred_final = clf_best.predict(X_test)
y_test_proba_final = clf_best.predict_proba(X_test)

test_metrics = {
    'accuracy': accuracy_score(y_test, y_test_pred_final),
    'f1_macro': f1_score(y_test, y_test_pred_final, average='macro'),
    'roc_auc': roc_auc_score(y_test, y_test_proba_final, multi_class='ovr', average='macro'),
}
pd.DataFrame([test_metrics]).to_csv(RESULTS_DIR / "test_final_metrics.csv", index=False)

print(f"📊 FINAL TEST: Acc={test_metrics['accuracy']:.2%} | F1={test_metrics['f1_macro']:.2%} | AUC={test_metrics['roc_auc']:.2%}")
print(classification_report(y_test, y_test_pred_final, 
                            target_names=['Früh (<1J)', 'Spät (>1J)', 'Gesund']))

Best config: ensemble_8, seed: 49


In [23]:
print(f"📊 FINAL TEST: Acc={test_metrics['accuracy']:.2%} | F1={test_metrics['f1_macro']:.2%} | AUC={test_metrics['roc_auc']:.2%}")
print(classification_report(y_test, y_test_pred_final, 
                            target_names=['Früh (<1J)', 'Spät (>1J)', 'Gesund']))

📊 FINAL TEST: Acc=59.71% | F1=38.69% | AUC=81.00%
              precision    recall  f1-score   support

  Früh (<1J)       0.15      0.60      0.24      2148
  Spät (>1J)       0.10      0.73      0.18      1630
      Gesund       0.98      0.59      0.74     40913

    accuracy                           0.60     44691
   macro avg       0.41      0.64      0.39     44691
weighted avg       0.90      0.60      0.69     44691



# 7. Feature Importance

In [24]:
from sklearn.inspection import permutation_importance
import pandas as pd
import plotly.express as px

if skip:
    print("skipped, dauert zu lang der Scheiss")
else:
    # Bestes Modell aus Robustness Loop verwenden (nicht den alten classifier!)
    print("Berechne Feature Importance mit bestem Modell (n_repeats=10)...")

    result = permutation_importance(
        clf_best,          # ← bestes Modell aus Section 6. Test
        X_val,             # ← volles Val-Set, nicht Sample!
        y_val,
        n_repeats=10,      # ← Standard für Thesis
        random_state=42, 
        scoring='f1_macro',  # ← passend zu deiner Hauptmetrik
        n_jobs=1
    )

    importance_df = pd.DataFrame({
        'Feature': X_val.columns,  # ← dynamisch statt hardcoded
        'Importance': result.importances_mean,
        'Std_Dev': result.importances_std
    }).sort_values(by='Importance', ascending=True)
    
    # Speichern für Thesis
    importance_df.to_csv(RESULTS_DIR / "feature_importance.csv", index=False)

    fig = px.bar(
        importance_df, 
        x='Importance', 
        y='Feature', 
        orientation='h',
        title='Feature Importance (Permutation, n=10)',
        labels={'Importance': 'Mean Decrease in F1 Macro'},
        error_x='Std_Dev',
        template="plotly_white",
        color='Importance',
        color_continuous_scale='Reds'
    )

    fig.update_layout(height=500, xaxis_tickformat='.1%')
    show_and_save(fig, "feature_importance")

Berechne Feature Importance mit bestem Modell (n_repeats=10)...
💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260303_212527/plots/feature_importance.png
